### Installation

In [2]:
!pip install unsloth vllm
# Temporarily install a specific TRL nightly version

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.3/196.3 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.3/265.3 MB 6.4 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 2.2 MB/s eta 0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.7/766.7 MB 2.0 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 68.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 94.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.1/253.1 MB 1.9 MB/s eta 0:00:00:

In [3]:
!pip install trl

In [4]:
!pip install triton==3.1.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.5/209.5 MB 8.1 MB/s eta 0:00:000:00:0100:01
  Attempting uninstall: triton
    Found existing installation: triton 3.2.0
    Uninstalling triton-3.2.0:
      Successfully uninstalled triton-3.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0 requires triton==3.2.0; platform_system == "Linux" and platform_machine == "x86_64", but you have triton 3.1.0 which is incompatible.


In [5]:
!pip install -U pynvml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.4 MB/s eta 0:00:00
  Attempting uninstall: pynvml
    Found existing installation: pynvml 11.4.1
    Uninstalling pynvml-11.4.1:
      Successfully uninstalled pynvml-11.4.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 24.12.0 requires pynvml<12.0.0a0,>=11.0.0, but you have pynvml 12.0.0 which is incompatible.
dask-cudf-cu12 24.12.0 requires pynvml<12.0.0a0,>=11.4.1, but you have pynvml 12.0.0 which is incompatible.
rapids-dask-dependency 24.12.0 requires pynvml<11.5.0a0,>=11.0.0, but you have pynvml 12.0.0 which is incompatible.
ucx-py-cu12 0.41.0 requires pynvml<12.0.0a0,>=11.4.1, but you have pynvml 12.0.0 which is incompatible.
ucxx-cu12 0.41.0 requires pynvml<12.0.0a0,>=11.4.1, but you have pynvml 12.0.0 which is incompatible.


### Unsloth

Use `PatchFastRL` before all functions to patch GRPO and other RL algorithms!

In [6]:
from unsloth import FastLanguageModel, PatchFastRL, is_bfloat16_supported

PatchFastRL("GRPO", FastLanguageModel)

from transformers import Trainer, TrainingArguments

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 03-19 17:11:06 [__init__.py:256] Automatically detected platform cuda.


Load up `Llama 3.1 8B Instruct`, and set parameters

In [7]:
max_seq_length = 512  # Can increase for longer reasoning traces
lora_rank = 32  # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="meta-llama/meta-Llama-3.1-8B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,  # False for LoRA 16bit
    fast_inference=True,  # Enable vLLM fast inference
    max_lora_rank=lora_rank,
    gpu_memory_utilization=0.6,  # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,  # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],  # Remove QKVO if out of memory
    lora_alpha=lora_rank,
    use_gradient_checkpointing="unsloth",  # Enable long context finetuning
    random_state=3407,
)

==((====))==  Unsloth 2025.3.17: Fast Llama patching. Transformers: 4.49.0. vLLM: 0.8.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/meta-llama-3.1-8b-instruct-unsloth-bnb-4bit with actual GPU utilization = 59.43%
Unsloth: Your GPU has CUDA compute capability 7.5 with VRAM = 14.74 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 512. Num Sequences = 160.
Unsloth: vLLM's KV Cache can use up to 2.59 GB. Also swap space = 5 GB.
WARNING 03-19 17:12:50 [config.py:2599] Casting torch.bfloat16 to torch.float16.
INFO 03-19 17:13:02 [config.py:583] This model supports multiple tasks: {'embed', 'reward', 'score', 'ge

tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

...

INFO 03-19 17:15:04 [model_runner.py:1570] Graph capturing finished in 49 secs, took 0.53 GiB
INFO 03-19 17:15:04 [llm_engine.py:447] init engine (profile, create kv cache, warmup model) took 73.87 seconds


tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

Unsloth 2025.3.17 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


### Data Prep
<a name="Data"></a>

We directly leverage [@willccbb](https://gist.github.com/willccbb/4676755236bb08cab5f4e54a0475d6fb) for data prep and all reward functions. You are free to create your own!

In [34]:
import os

# List all files in the input directory
input_dir = "/kaggle/input/"

# Loop through the directory to print dataset names
for root, dirs, files in os.walk(input_dir):
    print(f"📂 Directory: {root}")
    for dir_name in dirs:
        print(f"   📁 Dataset: {os.path.join(root, dir_name)}")
    for file_name in files:
        print(f"   📄 File: {os.path.join(root, file_name)}")
    print("-" * 50)

📂 Directory: /kaggle/input/
   📁 Dataset: /kaggle/input/star-wars
   📁 Dataset: /kaggle/input/star-wars3
   📁 Dataset: /kaggle/input/star-wars2-json
--------------------------------------------------
📂 Directory: /kaggle/input/star-wars
   📄 File: /kaggle/input/star-wars/star-wars
--------------------------------------------------
📂 Directory: /kaggle/input/star-wars3
   📄 File: /kaggle/input/star-wars3/star-wars.json
--------------------------------------------------
📂 Directory: /kaggle/input/star-wars2-json
   📄 File: /kaggle/input/star-wars2-json/star-wars.json
--------------------------------------------------


In [36]:
import os
from datasets import load_dataset

print(os.getcwd())
file_path = "/kaggle/input/star-wars3/star-wars.json"

/kaggle/working


In [37]:
dataset = load_dataset("json", data_files={"train": file_path})["train"]
display(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['Character', 'Line'],
    num_rows: 2523
})

In [38]:
dataset = load_dataset("json", data_files={"train": file_path})["train"]


# Preprocessing: Combine the character and line into one string.
def preprocess(example):
    # Create a dialogue-like string (e.g., "THREEPIO: We're doomed!")
    text = f"{example['Character']}: {example['Line']}"
    # Tokenize the text; we use padding and truncation to ensure uniform length.
    tokenized = tokenizer(
        text, truncation=True, max_length=max_seq_length, padding="max_length"
    )
    # For language modeling, labels are the same as the input IDs.
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized


# Apply the preprocessing to the entire dataset.
tokenized_dataset = dataset.map(preprocess, batched=False)

Map:   0%|          | 0/2523 [00:00<?, ? examples/s]

In [39]:
display(tokenized_dataset)

Dataset({
    features: ['Character', 'Line', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 2523
})

In [41]:
# Define training arguments using Hugging Face's Trainer API.
training_args = TrainingArguments(
    output_dir="./star_wars_model",  # Directory to save model checkpoints
    per_device_train_batch_size=1,  # Adjust based on your GPU memory
    gradient_accumulation_steps=1,  # Increase for smoother training if needed
    num_train_epochs=2,  # Set epochs based on your dataset size
    learning_rate=5e-6,  # Adjust the learning rate as needed
    weight_decay=0.1,
    logging_steps=10,  # Log every 10 steps
    save_steps=500,  # Save a checkpoint every 500 steps
    fp16=not is_bfloat16_supported(),  # Use FP16 if BF16 is not supported
    bf16=is_bfloat16_supported(),
    report_to="none",
)

In [43]:
# Initialize the Trainer; this automatically uses cross-entropy loss for language modeling.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
)

# Start training
trainer.train()

<ipython-input-43-f7a520dc5ae8>:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,523 | Num Epochs = 2 | Total steps = 2,524
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 1 x 1) = 2
 "-____-"     Trainable parameters = 83,886,080/8,000,000,000 (1.05% trained)


Step,Training Loss
10,11.305500
20,11.374100
30,11.022900
40,10.976000
50,9.899000
60,9.077800
70,9.308500
80,8.114800
90,7.589500
100,7.579200


TrainOutput(global_step=2524, training_loss=5.990782527651538, metrics={'train_runtime': 10005.1206, 'train_samples_per_second': 0.504, 'train_steps_per_second': 0.252, 'total_flos': 6.49075618700329e+16, 'train_loss': 5.990782527651538, 'epoch': 2.0})

<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |


<a name="Inference"></a>
### Inference
Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

In [45]:
from vllm import SamplingParams

text = tokenizer.apply_chat_template(
    [
        {
            "role": "user",
            "content": "Tell me a bedtime story with Yoda's voice",
        },
    ],
    tokenize=False,
    add_generation_prompt=True,
)


sampling_params = SamplingParams(
    temperature=0.8,
    top_p=0.95,
    max_tokens=1024,
)
output = (
    model.fast_generate(
        [text],
        sampling_params=sampling_params,
        lora_request=None,
    )[0]
    .outputs[0]
    .text
)

output

Processed prompts: 100%|██████████| 1/1 [00:25<00:00, 25.81s/it, est. speed input: 1.78 toks/s, output: 14.45 toks/s]


'"A bedtime story, I shall tell you. Cozy and snug, you must get. \n\nIn a galaxy far, far away, a youngling named Luna lived. On a planet of dreams, she resided. Peaceful, the planet was. Lush and green, the forests were. Gentle, the creatures were.\n\nLuna, a curious one, she was. Explore the planet, she loved to. One day, a strange object, she discovered. A small, glowing stone, it was. In the forest, it lay.\n\n\'Take it, I must,\' said a wise old owl, perched on a nearby branch. \'A special power, it holds. But beware, young one, the dark side, it can bring.\'\n\nLuna, unsure, she was. But the owl\'s words, they resonated. The stone, she took. Its power, she felt. Strong and free, she felt.\n\nA journey, she embarked on. Through the forest, she walked. Dark and mysterious, the path was. But the stone\'s power, it guided her. Hidden temples, she found. Ancient secrets, she uncovered.\n\nBut a dark force, it threatened. A great storm, it brewed. The planet, it shook. Luna, she tremb

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [48]:
model.save_pretrained("/kaggle/working/star_wars_model_finetuned")

import os

print(os.listdir("/kaggle/working/"))  # Lists files in the working directory
print(os.listdir("/kaggle/input/"))  # Lists datasets you've added to the notebook

['huggingface_tokenizers_cache', 'star_wars_model', 'unsloth_compiled_cache', '.virtual_documents', 'star_wars_model_finetuned']
['star-wars', 'star-wars3', 'star-wars2-json']


Now we load the LoRA and test:

In [ ]:
text = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Calculate pi."},
    ],
    tokenize=False,
    add_generation_prompt=True,
)

from vllm import SamplingParams

sampling_params = SamplingParams(
    temperature=0.8,
    top_p=0.95,
    max_tokens=1024,
)
output = (
    model.fast_generate(
        text,
        sampling_params=sampling_params,
        lora_request=model.load_lora("grpo_saved_lora"),
    )[0]
    .outputs[0]
    .text
)

output

<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False:
    model.save_pretrained_merged(
        "model",
        tokenizer,
        save_method="merged_16bit",
    )
if False:
    model.push_to_hub_merged(
        "hf/model", tokenizer, save_method="merged_16bit", token=""
    )

# Merge to 4bit
if False:
    model.save_pretrained_merged(
        "model",
        tokenizer,
        save_method="merged_4bit",
    )
if False:
    model.push_to_hub_merged("hf/model", tokenizer, save_method="merged_4bit", token="")

# Just LoRA adapters
if False:
    model.save_pretrained_merged(
        "model",
        tokenizer,
        save_method="lora",
    )
if False:
    model.push_to_hub_merged("hf/model", tokenizer, save_method="lora", token="")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/drive/1WZDi7APtQ9VsvOrQSSC5DDtxq159j8iZ?usp=sharing)

In [ ]:
# Save to 8bit Q8_0
if False:
    model.save_pretrained_gguf(
        "model",
        tokenizer,
    )
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False:
    model.push_to_hub_gguf("hf/model", tokenizer, token="")

# Save to 16bit GGUF
if False:
    model.save_pretrained_gguf("model", tokenizer, quantization_method="f16")
if False:
    model.push_to_hub_gguf("hf/model", tokenizer, quantization_method="f16", token="")

# Save to q4_k_m GGUF
if False:
    model.save_pretrained_gguf("model", tokenizer, quantization_method="q4_k_m")
if False:
    model.push_to_hub_gguf(
        "hf/model", tokenizer, quantization_method="q4_k_m", token=""
    )

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model",  # Change hf to your username!
        tokenizer,
        quantization_method=[
            "q4_k_m",
            "q8_0",
            "q5_k_m",
        ],
        token="",
    )